In [1]:
!pip uninstall -y torchvision -q
!pip install -q datasets transformers accelerate evaluate scikit-learn pandas numpy joblib

In [2]:
import os
import re
import time
import random
import shutil
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, concatenate_datasets, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ["WANDB_DISABLED"] = "true"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [4]:
ds = load_dataset("contemmcm/sentiment140")

ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    complete: Dataset({
        features: ['text', 'label'],
        num_rows: 1600000
    })
})

In [5]:
data = ds["complete"]

data[0]

{'text': "Just back home from a little gathering with some old friends.. It was really fun, they're still the same. ",
 'label': 1}

In [6]:
SUBSAMPLE_SIZE = 50_000

In [7]:
negative = data.filter(lambda x: x["label"] == 0)
positive = data.filter(lambda x: x["label"] == 1)

n_per_class = SUBSAMPLE_SIZE // 2

negative_sample = negative.shuffle(seed=SEED).select(range(n_per_class))
positive_sample = positive.shuffle(seed=SEED).select(range(n_per_class))

balanced_data = concatenate_datasets([negative_sample, positive_sample]).shuffle(seed=SEED)

print("Balanced sample size:", len(balanced_data))

Balanced sample size: 50000


In [8]:
df = pd.DataFrame({
    "text": balanced_data["text"],
    "label": [0 if y == 0 else 1 for y in balanced_data["label"]]
})

df.head()

,text,label
0,I'm actually pretty sad the Duke has left us ...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...",0
2,@christina1986 what cam did you get now? the ...,1
3,ha! one line css grids framework. http://bit....,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...",0


In [9]:
df["label"].value_counts()

,count
label,
0,25000
1,25000


In [10]:
def clean_tweet(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Replace @usernames with @user
    text = re.sub(r"@\w+", "@user", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_text"] = df["text"].apply(clean_tweet)

df[["text", "clean_text", "label"]].head(10)

,text,clean_text,label
0,I'm actually pretty sad the Duke has left us ...,I'm actually pretty sad the Duke has left us W...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...","@user haha spazzz ;) and i know, i've only jus...",0
2,@christina1986 what cam did you get now? the ...,@user what cam did you get now? the one that i...,1
3,ha! one line css grids framework. http://bit....,ha! one line css grids framework. (via @user) ...,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...","GOODMORNING... Ugh, twitter fully kicked me of...",0
5,Itchy eyes,Itchy eyes,0
6,off to school tom. which is soo not good. i ha...,off to school tom. which is soo not good. i ha...,0
7,"@cdncamel LOL, Claire. Even if you pass on th...","@user LOL, Claire. Even if you pass on the roo...",1
8,Really feel like crap this morning feel like ...,Really feel like crap this morning feel like I...,0
9,@peterloggins hehe i know! Luka is the BEST fr...,@user hehe i know! Luka is the BEST friend ever!,1


In [11]:
X = df["clean_text"].tolist()
y = df["label"].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 40000
Test size: 10000


In [12]:
train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset, test_dataset

(Dataset({
     features: ['text', 'label'],
     num_rows: 40000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 10000
 }))

In [13]:
train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset, test_dataset

(Dataset({
     features: ['text', 'label'],
     num_rows: 40000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 10000
 }))

In [14]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [15]:
MAX_LENGTH = 64

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_dataset.map(tokenize_batch, batched=True)
test_tokenized = test_dataset.map(tokenize_batch, batched=True)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [16]:
print("Train columns:", train_tokenized.column_names)
print("Test columns:", test_tokenized.column_names)



Train columns: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
Test columns: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [17]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [20]:
training_args = TrainingArguments(
    output_dir="./distilbert_sentiment140_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [22]:
start_time = time.time()

trainer.train()

train_time = time.time() - start_time

print(f"Training time: {train_time:.2f} seconds")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.405641,0.388668,0.826100,0.821183
2,0.275997,0.416304,0.829500,0.825218


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time: 283.38 seconds


In [23]:
eval_results = trainer.evaluate()

eval_results

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.275997,0.416304,2,0.829500,0.825218


{'eval_loss': 0.4163035750389099,
 'eval_accuracy': 0.8295,
 'eval_f1': 0.8252178370066633}

In [24]:
pred_output = trainer.predict(test_tokenized)

logits = pred_output.predictions
y_pred = np.argmax(logits, axis=1)
y_true = pred_output.label_ids

test_accuracy = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test F1: {test_f1:.4f}")

Test accuracy: 0.8295
Test F1: 0.8252


In [25]:
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"]
))

              precision    recall  f1-score   support

    negative       0.81      0.85      0.83      5000
    positive       0.85      0.81      0.83      5000

    accuracy                           0.83     10000
   macro avg       0.83      0.83      0.83     10000
weighted avg       0.83      0.83      0.83     10000



In [26]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

cm_df

,Predicted Negative,Predicted Positive
Actual Negative,4270,730
Actual Positive,975,4025


In [27]:
model.eval()

test_texts = X_test

start_time = time.time()

for text in test_texts:
    clean = clean_tweet(text)

    inputs = tokenizer(
        clean,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        _ = model(**inputs)

total_inference_time = time.time() - start_time
latency_per_example_ms = (total_inference_time / len(test_texts)) * 1000

print(f"Total inference time: {total_inference_time:.4f} seconds")
print(f"Inference latency: {latency_per_example_ms:.4f} ms/example")

Total inference time: 71.1662 seconds
Inference latency: 7.1166 ms/example


In [28]:
save_dir = "distilbert_sentiment140_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved model to:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: distilbert_sentiment140_model


In [29]:
def get_dir_size_mb(path):
    total_size = 0

    for dirpath, dirnames, filenames in os.walk(path):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)
            total_size += os.path.getsize(file_path)

    return total_size / (1024 * 1024)

model_size_mb = get_dir_size_mb(save_dir)

print(f"Model size: {model_size_mb:.2f} MB")

Model size: 256.11 MB


In [30]:
transformer_results = {
    "Model": "DistilBERT fine-tune",
    "Dataset Size": len(df),
    "Train Size": len(X_train),
    "Test Size": len(X_test),
    "Accuracy": test_accuracy,
    "F1": test_f1,
    "Train Time (s)": train_time,
    "Inference (ms/example)": latency_per_example_ms,
    "Model Size (MB)": model_size_mb,
    "Max Tokens": MAX_LENGTH,
    "Epochs": 2
}

transformer_results_df = pd.DataFrame([transformer_results])

transformer_results_df

,Model,Dataset Size,Train Size,Test Size,Accuracy,F1,Train Time (s),Inference (ms/example),Model Size (MB),Max Tokens,Epochs
0,DistilBERT fine-tune,50000,40000,10000,0.8295,0.825218,283.382543,7.116618,256.109494,64,2


In [31]:
transformer_results_df.to_csv("transformer_model_results.csv", index=False)

print("Saved results to transformer_model_results.csv")

Saved results to transformer_model_results.csv
